In [1]:
# ============================================
# Cell 1: Импорт библиотек
# ============================================
# pandas - для работы с таблицами CSV (вопросы и ответы)
# numpy - для числовых операций (сортировка, индексы)
# re - регулярные выражения для очистки HTML
# pickle - для сохранения/загрузки бинарных файлов
# sentence_transformers - готовая модель для эмбеддингов текста
# sklearn.metrics.pairwise - косинусная близость между векторами

import pandas as pd
import numpy as np
import re
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re

C:\Users\1\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================
# Cell 2: СОЗДАНИЕ questions_small через chunks
# ============================================
# ПОЧЕМУ chunk'и?
# Файл Questions.csv весит 1,79 гигабайт.
# Чтение целиком вызовет MemoryError на моем компьютере.
# Поэтому читаем по 50 000 строк, фильтруем Score > 0 (хорошие вопросы)
# и останавливаемся, когда набрали 5000 вопросов.

questions_small_list = []
chunksize = 50000  # Читаем по 50 000 строк
needed = 5000      # Нужно набрать 5000 вопросов

for chunk in pd.read_csv(
     "Questions.csv",
    encoding='latin-1',
    chunksize=chunksize
):
    # Фильтруем хорошие вопросы (Score > 0)
    filtered = chunk[chunk['Score'] > 0]
    questions_small_list.append(filtered)
    
    # Проверяем, сколько уже набрали
    total = sum(len(df) for df in questions_small_list)
    
    if total >= needed:
        break

# Объединяем и берем ровно needed
questions_small = pd.concat(questions_small_list, ignore_index=True).head(needed)

In [ ]:
# ============================================
# Cell 3: Очистка текста вопросов
# ============================================
# В Body могут быть HTML-теги (<p>, <code>, <a>), которые мешают поиску.
# Также есть inline-код в обратных кавычках `print('hello')` — он не нужен,
# потому что пользователь спрашивает про смысл, а не про конкретный код.
print("🧹 Очистка текста вопросов...")

# Приводим к строкам
questions_small['Title'] = questions_small['Title'].astype(str)
questions_small['Body'] = questions_small['Body'].astype(str)

# Удаляем HTML теги
questions_small['Body'] = questions_small['Body'].str.replace(r'<[^>]+>', ' ', regex=True)

# Удаляем inline code
questions_small['Body'] = questions_small['Body'].str.replace(r'`.*?`', ' ', regex=True)

# Убираем лишние пробелы
questions_small['Body'] = questions_small['Body'].str.replace(r'\s+', ' ', regex=True)

# Склеиваем Title и Body для поиска.
# Почему вместе? Потому что заголовок часто короткий и точный,
# а тело даёт контекст. Вместе они дают лучший семантический поиск.
questions_small['search_text'] = "Question: " + questions_small['Title'] + "\n\n" + questions_small['Body']

print("✅ Текст очищен")

In [ ]:
# ============================================
# Cell 4: Создание эмбеддингов для вопросов
# ============================================
# ПОЧЕМУ all-MiniLM-L6-v2?
# 1. Лёгкая (384 dim) — работает даже на CPU.
# 2. Качество близко к большим моделям (MPNet, BERT large).
# 3. Скорость: 1000 вопросов ~ 10-15 секунд.
# 4. Специально обучена на семантическое сходство (SBERT).
print("🤖 Загрузка модели и создание эмбеддингов...")

model = SentenceTransformer('all-MiniLM-L6-v2')

# Превращаем каждый вопрос в вектор длиной 384
question_texts = questions_small['search_text'].tolist()
embeddings = model.encode(question_texts, show_progress_bar=True)

print(f"✅ Эмбеддинги созданы! Форма: {embeddings.shape}")

In [ ]:
# ============================================
# Cell 5: Сохранение вопросов и эмбеддингов
# ============================================
# ПОЧЕМУ pickle?
# pickle сохраняет Python-объекты как есть (DataFrame, numpy array).
# Это быстро и удобно для загрузки в Streamlit.

print("💾 Сохранение файлов...")

with open('questions_small.pkl', 'wb') as f:
    pickle.dump(questions_small, f)

with open('question_embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

print("✅ questions_small.pkl сохранен")
print("✅ question_embeddings.pkl сохранен")

In [ ]:
# ============================================
# Cell 6: Функция очистки HTML для ответов
# ============================================
# В Answers.csv тело ответа тоже содержит HTML и сущности (&gt; = > и т.д.)
# Без очистки ответ будет нечитаем: "&lt;div&gt;Hello&lt;/div&gt;"


def clean_html(text):
    """Очищает текст от HTML-тегов и заменяет HTML-сущности"""
    if not isinstance(text, str):
        return ""
    
    # Заменяем HTML-сущности
    text = text.replace('&gt;', '>')
    text = text.replace('&lt;', '<')
    text = text.replace('&amp;', '&')
    text = text.replace('&quot;', '"')
    text = text.replace('&#39;', "'")
    
    # Удаляем HTML-теги
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Убираем лишние пробелы и пустые строки
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

print("✅ Функция очистки HTML готова")

In [ ]:
# ============================================
# Cell 7: СОЗДАНИЕ ФИЛЬТРОВАННОГО СЛОВАРЯ ОТВЕТОВ
# ============================================
# ПОЧЕМУ словарь, а не DataFrame?
# Для быстрого доступа: answers[question_id] -> (текст_ответа, рейтинг)
# Это O(1) вместо поиска по DataFrame.
#
# ПОЧЕМУ только лучший ответ на вопрос?
# На один вопрос может быть 10+ ответов. Нам нужен самый полезный,
# поэтому берём ответ с максимальным Score (рейтинг).
#
# ПОЧЕМУ chunks в Answers.csv?
# Answers.csv 1,49 ГБ
print("📚 Создание фильтрованного словаря ответов (только для наших вопросов)...")

# Множество ID вопросов из нашей выборки
# Множество даёт проверку membership за O(1)
question_ids = set(questions_small['Id'].tolist())
print(f"  Вопросов в выборке: {len(question_ids)}")

# Создаем фильтрованный словарь
filtered_answers = {}

for chunk in pd.read_csv(
    "Answers.csv",
    encoding='latin-1',
    chunksize=100000,
    usecols=['ParentId', 'Score', 'Body']
):
    # Оставляем только ответы на наши вопросы
    chunk_filtered = chunk[chunk['ParentId'].isin(question_ids)]
    
    for _, row in chunk_filtered.iterrows():
        parent_id = row['ParentId']
        score = row['Score']
        body = clean_html(row['Body'])  # Очищаем HTML прямо при загрузке
        
        #Если ответа ещё нет или новый ответ имеет больший рейтинг — сохраняем
        if parent_id not in filtered_answers or filtered_answers[parent_id][1] < score:
            filtered_answers[parent_id] = (body, score)

print(f"✅ Найдено ответов для {len(filtered_answers)} вопросов из {len(question_ids)}")

In [ ]:
# ============================================
# Cell 8: Сохранение фильтрованного словаря
# ============================================
print("💾 Сохранение фильтрованного словаря...")

with open('filtered_answers.pkl', 'wb') as f:
    pickle.dump(filtered_answers, f)

print("✅ filtered_answers.pkl сохранен")

# Очищаем память
del filtered_answers
import gc
gc.collect()
print("🧹 Память очищена")

In [3]:
# ============================================
# Cell 9: ДЕМОНСТРАЦИЯ — один случайный пример
# ============================================

import random

# Загружаем данные
with open('questions_small.pkl', 'rb') as f:
    questions = pickle.load(f)

with open('filtered_answers.pkl', 'rb') as f:
    answers = pickle.load(f)

# Собираем вопросы, у которых есть ответы
questions_with_answers = [(idx, row) for idx, row in questions.iterrows() if row['Id'] in answers]

# Выбираем случайный
idx, row = random.choice(questions_with_answers)
q_id = row['Id']
answer_text, answer_score = answers[q_id]

# Выводим
print("="*70)
print("📌 СЛУЧАЙНЫЙ ВОПРОС ИЗ ДАТАСЕТА")
print("="*70)

print(f"\n❓ ВОПРОС (рейтинг: {row['Score']})")
print(f"   {row['Title']}")

print(f"\n💡 ОТВЕТ (рейтинг: {answer_score})")
print(f"   {answer_text[:500]}...")

print("\n" + "="*70)

📌 СЛУЧАЙНЫЙ ВОПРОС ИЗ ДАТАСЕТА

❓ ВОПРОС (рейтинг: 11)
   How to get Django AutoFields to start at a higher number

💡 ОТВЕТ (рейтинг: 10)
   Like the others have said, this would be much easier to do on the database side than the Django side. For Postgres, it'd be like so : ALTER SEQUENCE sequence_name RESTART WITH 12345; Look at your own DB engine's docs for how you'd do it there....



In [4]:
# ============================================
# Cell 12: ИНСТРУКЦИЯ ПО ЗАПУСКУ
# ============================================

print("""
🚀 ЗАПУСК ПРИЛОЖЕНИЯ:

1. Откройте терминал в папке проекта

2. Выполните команду:
   python -m streamlit run app.py

3. Дождитесь запуска сервера
   → Откроется браузер с адресом http://localhost:8501

4. Задайте вопрос в текстовое поле и нажмите Search

✅ Готово!
""")


🚀 ЗАПУСК ПРИЛОЖЕНИЯ:

1. Откройте терминал в папке проекта

2. Выполните команду:
   python -m streamlit run app.py

3. Дождитесь запуска сервера
   → Откроется браузер с адресом http://localhost:8501

4. Задайте вопрос в текстовое поле и нажмите Search

✅ Готово!

